# Span-Free Information Extraction with GLiNER2.5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/aibackends/blob/main/examples/notebooks/gliner25_information_extraction_colab.ipynb)

GLiNER2.5 is a schema-conditioned encoder for local entity extraction, classification,
relations, structured records, and per-span attributes. Its boundary architecture predicts
entity starts and ends instead of enumerating fixed-width spans.

**What this notebook covers**

1. Load one of the small, base, or multilingual checkpoints
2. Extract ordinary and clause-length entities with source offsets
3. Scan a full document with overlapping chunks and global offsets
4. Route agents with constrained classification
5. Build a typed knowledge graph with Joint IE
6. Attach clinical attributes to entity spans
7. Run a combined multi-task schema in one pass
8. Use native batch extraction

> The `small` checkpoint is the practical default for a free CPU runtime. Choose `base` for
> stronger English extraction or `multi` for multilingual documents.

## Setup

Install the `aibackends` GLiNER2.5 capability extra. The notebook uses the first-class
information-extraction backend throughout; the native model dependency stays behind that API.

In [ ]:
!pip install -q "aibackends[gliner2]"

# If a later import fails after installation, use Runtime > Restart session,
# then continue from the next cell.

In [ ]:
import json
import time

import aibackends
import torch
from aibackends.backends.information_extraction import (
    get_information_extraction_backend,
    list_information_extraction_backends,
)
from aibackends.backends.information_extraction.gliner25 import (
    GLINER25_MODEL_IDS,
    assert_source_spans,
    result_to_dict,
)
from aibackends.tasks import (
    batch_extract_entities,
    classify_schema,
    extract_entities,
    extract_entities_long,
    extract_graph,
    extract_schema,
)

MODEL = "small"  # @param ["small", "base", "multi"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
backend = get_information_extraction_backend("gliner25")
MODEL_ID = GLINER25_MODEL_IDS[MODEL]


def show(value):
    print(json.dumps(result_to_dict(value), indent=2, ensure_ascii=False))


def verify_spans(text, value):
    return assert_source_spans(text, value)


print("aibackends:", aibackends.__version__)
print("information-extraction backends:", list_information_extraction_backends())
print("model:", MODEL_ID)
print("device:", DEVICE)

## 1. Load once through `aibackends`

The `gliner25` information-extraction backend resolves the model alias, selects the native
boundary loader, and caches one model per checkpoint and device. The first run downloads model
weights; every task below reuses the backend cache.

In [ ]:
t = time.perf_counter()
backend.load(model=MODEL, device=DEVICE)
load_s = time.perf_counter() - t
print(f"aibackends backend load: {load_s:.1f}s")

## 2. Ordinary and clause-length entity extraction

Boundary prediction has no fixed maximum entity width. Request spans while developing so every
`[start, end)` pair can be checked directly against the source text.

In [ ]:
text = (
    "Acme Corporation appointed Maya Chen as chief scientist. "
    "The supplier must replace every defective battery within thirty calendar days "
    "after receiving written notice from the customer."
)
labels = {
    "organization": "Company or organization names",
    "person": "Names of people",
    "obligation": "A complete clause describing a required action",
}

t = time.perf_counter()
entities = extract_entities(
    text,
    labels,
    backend=backend.name,
    model=MODEL,
    device=DEVICE,
    include_spans=True,
    include_confidence=True,
)
warm_ms = (time.perf_counter() - t) * 1000

print("verified spans:", verify_spans(text, entities))
print(f"warm extraction: {warm_ms:.0f}ms (vs {load_s * 1000:.0f}ms to load)\n")
show(entities)

## 3. Full-document extraction with global offsets

The `*_long` APIs scan overlapping word chunks, remap local spans to document offsets, and merge
overlap duplicates. They avoid the silent truncation caused by passing a short `max_len`.

In [ ]:
boilerplate = (
    "The parties reviewed the schedules, definitions, notices, and standard administrative "
    "terms before signing. "
)
long_document = (
    "SERVICE AGREEMENT\nProvider: Maya Chen, maya@example.test, +1 555 0100.\n"
    + boilerplate * 30
    + "The Provider must delete all customer backups within forty-five days after termination.\n"
    + boilerplate * 30
    + "Either party may terminate this Agreement by giving ninety days written notice."
)

long_result = extract_entities_long(
    long_document,
    {
        "person": "Names of contract parties",
        "email": "Email addresses",
        "phone_number": "Telephone numbers",
        "obligation": "Complete clauses describing a required action",
        "termination_clause": "Complete clauses describing how the agreement can end",
    },
    backend=backend.name,
    model=MODEL,
    device=DEVICE,
    chunk_size=96,
    chunk_overlap=24,
    include_spans=True,
    include_confidence=True,
)
print("document words:", len(long_document.split()))
print("verified global spans:", verify_spans(long_document, long_result))
show(long_result)

## 4. Constrained model and agent routing

`classify_text` decodes tasks independently. `Classifier` instead searches for a joint assignment
that satisfies declared implications and exclusions, so an inferred task cannot select an
incompatible route.

In [ ]:
C = backend.classification_constraints
routing_schema = (
    backend.create_classification_schema()
    .single("task_type", ["summarization", "reasoning", "live_data"])
    .single("route", ["small_local_model", "large_reasoning_model", "tool_agent"])
    .constrain(
        C.implies(("task_type", "summarization"), ("route", "small_local_model")),
        C.implies(("task_type", "reasoning"), ("route", "large_reasoning_model")),
        C.implies(("task_type", "live_data"), ("route", "tool_agent")),
    )
)
config = backend.create_classification_config(decoder="auto", on_infeasible="raise")

for request in [
    "Summarize the attached report.",
    "Prove whether this graph has a Hamiltonian cycle.",
    "Look up the current weather in Singapore.",
]:
    routed = classify_schema(
        request,
        routing_schema,
        backend=backend.name,
        model=MODEL,
        device=DEVICE,
        config=config,
    )
    print("\n", request)
    show(routed)

## 5. Joint Information Extraction for a coherent graph

`JointIE` chooses entity mentions and typed relations together. Endpoint types, uniqueness rules,
and graph constraints are enforced while decoding rather than repaired afterward.

In [ ]:
graph_schema = (
    backend.create_joint_schema(model=MODEL, device=DEVICE)
    .entities(["person", "organization", "location"])
    .relation("works_for", "person", "organization", unique_head=True)
    .relation("located_in", "organization", "location", unique_head=True)
    .no_self_loops()
)
graph_text = (
    "Tim Cook leads Apple in Cupertino. "
    "Sundar Pichai runs Google in Mountain View."
)
graph = extract_graph(
    graph_text,
    graph_schema,
    backend=backend.name,
    model=MODEL,
    device=DEVICE,
    config=backend.create_joint_config(optimizer="beam", beam_size=32),
)
print("feasible:", graph.feasible)
print("verified entity spans:", verify_spans(graph_text, graph))
for relation in graph.relations:
    head = graph.entity(relation.head)
    tail = graph.entity(relation.tail)
    print(f"{head.text} -{relation.type}-> {tail.text}")
show(graph)

## 6. Clinical extraction with span attributes

Attributes qualify each retained span in the same forward pass. Here, symptoms receive a
negation status and medications receive a dosage form; these are not document-level labels.

In [ ]:
clinical_text = (
    "Patient reports a severe headache but denies chest pain. "
    "She started one 400 mg ibuprofen tablet every six hours and has no nausea."
)
clinical_schema = (
    backend.create_schema(model=MODEL, device=DEVICE)
    .entities({
        "symptom": "Symptoms or clinical findings",
        "medication": "Medication or drug names",
        "dosage": "Medication dose amounts",
    })
    .entity_attributes({
        "negation_status": backend.create_attribute_group(
            ["present", "negated"],
            applies_to=["symptom"],
            qualify_labels=True,
        ),
        "dosage_form": backend.create_attribute_group(
            ["tablet", "capsule", "liquid", "injection", "unspecified"],
            applies_to=["medication"],
            qualify_labels=True,
        ),
    })
)
clinical = extract_schema(
    clinical_text,
    clinical_schema,
    backend=backend.name,
    model=MODEL,
    device=DEVICE,
    include_spans=True,
    include_confidence=True,
)
print("verified spans:", verify_spans(clinical_text, clinical))
show(clinical)

## 7. Combined schema: multiple tasks in one pass

A single schema can request entities, document classification, independent relations, and
structured records. This shares encoder work and returns one source-grounded result.

In [ ]:
combined_text = (
    "Apple CEO Tim Cook announced the iPhone 15 for $999 in Cupertino. "
    "Reviewers praised the camera and called the launch exciting."
)
combined_schema = (
    backend.create_schema(model=MODEL, device=DEVICE)
    .entities(["person", "company", "product", "location"])
    .classification("sentiment", ["positive", "negative", "neutral"])
    .classification("document_type", ["product_news", "review", "opinion"])
    .relations(["works_for", "announced_by", "located_in"])
    .structure("product")
    .field("name", dtype="str")
    .field("price", dtype="str")
    .field("feature", dtype="list")
)
combined = extract_schema(
    combined_text,
    combined_schema,
    backend=backend.name,
    model=MODEL,
    device=DEVICE,
    include_spans=True,
    include_confidence=True,
)
print("verified spans:", verify_spans(combined_text, combined))
show(combined)

## 8. Native batch extraction

When many documents share a schema, use the batch API instead of a Python inference loop. The
result list remains aligned with the input list.

In [ ]:
documents = [
    "Apple CEO Tim Cook spoke in Cupertino.",
    "Microsoft CEO Satya Nadella presented Copilot in Seattle.",
    "Amazon CEO Andy Jassy announced new AWS services.",
]

t = time.perf_counter()
batch = batch_extract_entities(
    documents,
    ["company", "person", "product", "location"],
    backend=backend.name,
    model=MODEL,
    device=DEVICE,
    batch_size=3,
    include_spans=True,
    include_confidence=True,
)
batch_s = time.perf_counter() - t
for document, result in zip(documents, batch):
    verify_spans(document, result)
print(f"{len(batch)} documents in {batch_s:.2f}s ({len(batch) / batch_s:.1f}/s)")
show({"results": batch})

## Choosing a checkpoint

- **`small` (74M, English):** fastest CPU and edge option; use it for prototypes and latency-sensitive workloads.
- **`base` (194M, English):** default for stronger English multi-task extraction.
- **`multi` (287M, multilingual):** use for multilingual documents and cross-language schemas.

All three published checkpoints use the boundary architecture and enable record and relation
heads. Benchmark latency and evaluate quality on your own schema before selecting one.

## Production limits to keep visible

- A boundary span can be arbitrarily long only when both endpoints fit in the same encoded chunk.
- Long-document relations require both endpoints in one chunk; no cross-chunk edge is invented.
- Use `include_spans=True` during rollout and verify offsets against the source.
- Check `feasible` before accepting constrained classification or Joint IE output.
- Tune thresholds, chunk size, overlap, and schema wording on labeled domain examples.

## Continue with the repository examples

The `aibackends` repository includes standalone scripts for long documents, constrained routing,
Joint IE, span attributes, and combined schemas under `examples/gliner25/`, plus reproducible
accuracy evals and CPU benchmarks for all three checkpoints.

```bash
pip install -e '.[gliner2]'
python3 -m examples.gliner25.long_context --model small --device cpu
python3 -m examples.gliner25.constrained_routing --model base --device cpu
python3 -m examples.gliner25.joint_information_extraction --model multi --device cpu
```

## Recap

```python
from aibackends.backends.information_extraction import get_information_extraction_backend
from aibackends.tasks import batch_extract_entities, extract_entities, extract_entities_long

backend = get_information_extraction_backend("gliner25")
backend.load(model="base", device="cpu")
extract_entities(text, labels, model="base", include_spans=True)
extract_entities_long(document, labels, model="base", chunk_size=384, chunk_overlap=64)
batch_extract_entities(documents, labels, model="base", batch_size=8)
```

Use `backend.create_classification_schema()` with `classify_schema()` for constrained routing,
`backend.create_joint_schema()` with `extract_graph()` for typed graphs, and
`backend.create_attribute_group()` with `extract_schema()` for per-span context.

**Links**

- GLiNER2.5 release article — https://fastino.ai/blog/gliner2-5-span-free-information-extraction
- Base model — https://huggingface.co/fastino/gliner2.5-base-v1
- Small model — https://huggingface.co/fastino/gliner2.5-small-v1
- Multilingual model — https://huggingface.co/fastino/gliner2.5-multi-v1
- GLiNER2 repository — https://github.com/fastino-ai/GLiNER2
- aibackends repository — https://github.com/donvito/aibackends